# Notebook 05 — Embedding e Ricerca Semantica

**Settimana:** 5
**Blocco:** 4.1 — Embedding e modelli di embedding
**Tempo stimato:** 3-3.5 ore

## Cosa costruirai

- Similarità coseno implementata da zero con numpy
- Motore di ricerca semantica su un corpus di 80 frasi senza alcun framework
- Visualizzazione 2D degli embedding con PCA per vedere i cluster semantici
- Comparativa diretta: ricerca semantica vs ricerca per parole chiave sulle stesse query
- **Sfida finale:** FAQ semantica — dato un archivio di domande frequenti, trova la più simile alla query dell'utente

## Prerequisiti

```bash
pip install openai numpy matplotlib scikit-learn python-dotenv
# Opzionale per UMAP:
pip install umap-learn
```

> **Nota sui costi:** questo notebook effettua chiamate all'API di embedding di OpenAI.
> Con `text-embedding-3-small` e i corpus di questo notebook (~5.000 token in totale)
> il costo è inferiore a **$0.001**. Completamente trascurabile.

---
## Cella 0 — Setup

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
cliente = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODELO_EMBEDDING = "text-embedding-3-small"

# ── Funzione base: genera l'embedding di uno o più testi ─────────────────────
def generar_embedding(texto: str) -> np.ndarray:
    """Genera l'embedding di un testo. Restituisce np.ndarray di shape (1536,)."""
    respuesta = cliente.embeddings.create(
        model=MODELO_EMBEDDING,
        input=texto
    )
    return np.array(respuesta.data[0].embedding)


def generar_embeddings_batch(textos: list[str]) -> np.ndarray:
    """
    Genera gli embedding di una lista di testi in un'unica chiamata API.
    Restituisce np.ndarray di shape (n_textos, 1536).
    Più efficiente che chiamare generar_embedding() in un ciclo.
    """
    respuesta = cliente.embeddings.create(
        model=MODELO_EMBEDDING,
        input=textos
    )
    # I risultati arrivano ordinati per indice
    vectores = [item.embedding for item in sorted(respuesta.data, key=lambda x: x.index)]
    return np.array(vectores)


# Test rapido
v_prueba = generar_embedding("Ciao mondo")
print(f"Dimensioni dell'embedding: {v_prueba.shape}")
print(f"Tipo di dati:              {v_prueba.dtype}")
print(f"Norma del vettore:         {np.linalg.norm(v_prueba):.4f}  (dovrebbe essere ~1.0)")

---
## Cella 1 — Similarità coseno da zero

Implementa la similarità coseno senza usare alcuna libreria di ML.
Poi usala per esplorare quali coppie di frasi il modello considera simili.

**Prima di eseguire:** guarda le frasi del gruppo di test e prevedi
quali avranno similarità > 0.85 e quali < 0.3.

In [ ]:
# ── Implementazione da zero ─────────────────────────────────────────────────

def similitud_coseno(a: np.ndarray, b: np.ndarray) -> float:
    """Similarità coseno tra due vettori. Intervallo: [-1, 1]."""
    norma_a = np.linalg.norm(a)
    norma_b = np.linalg.norm(b)
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norma_a * norma_b))


def similitudes_batch(query: np.ndarray, corpus: np.ndarray) -> np.ndarray:
    """
    Similarità tra un vettore query e tutti i vettori del corpus.
    Usa operazioni matriciali — molto più veloce di un ciclo.

    query:  shape (d,)
    corpus: shape (n, d)
    return: shape (n,)  — una similarità per documento
    """
    query_norm  = query  / np.linalg.norm(query)
    normas      = np.linalg.norm(corpus, axis=1, keepdims=True)  # shape (n, 1)
    corpus_norm = corpus / normas
    return corpus_norm @ query_norm   # prodotto matriciale: (n, d) @ (d,) = (n,)


# ── Gruppi di frasi per sperimentare ───────────────────────────────────────
# 4 gruppi tematici + 2 frasi anomale

FRASES = [
    # Gruppo A: cani
    "Il cane abbaia in giardino",
    "Fido ulula di notte",
    "Il mio amico a quattro zampe mangia le crocchette",
    "I canidi sono animali domestici fedeli",

    # Gruppo B: programmazione Python
    "Python è un linguaggio di programmazione interpretato",
    "Le liste in Python sono mutabili",
    "L'interprete di CPython esegue bytecode",
    "I decoratori sono una caratteristica avanzata di Python",

    # Gruppo C: cucina
    "La paella valenciana contiene riso e zafferano",
    "Il soffritto è la base di molte ricette mediterranee",
    "Preriscalda il forno a 180 gradi prima di infornare",
    "L'olio extravergine di oliva è il migliore per friggere",

    # Gruppo D: intelligenza artificiale
    "I modelli linguistici generano testo token per token",
    "Il fine-tuning adatta un modello preaddestrato a un compito specifico",
    "Gli embedding rappresentano il significato semantico in vettori",
    "L'attenzione multi-testa è il meccanismo centrale del Transformer",

    # Frasi anomale (senza gruppo chiaro)
    "Il numero 42 è la risposta a tutto",
    "Domani pioverà nel nord della penisola",
]

ETIQUETAS = (
    ["🐕 Cani"]   * 4 +
    ["🐍 Python"] * 4 +
    ["🍳 Cucina"] * 4 +
    ["🤖 IA"]     * 4 +
    ["❓ Altro"]  * 2
)

print(f"Generando {len(FRASES)} embedding...")
MATRIZ = generar_embeddings_batch(FRASES)   # shape (18, 1536)
print(f"Matrice di embedding: {MATRIZ.shape}")


# ── Esperimento 1: similarità tra coppie specifiche ────────────────────────
print("\n=== Similarità tra coppie ===")
pares = [
    (0, 1, "cane ↔ fido"),
    (0, 4, "cane ↔ Python"),
    (4, 5, "Python interpretato ↔ liste Python"),
    (12, 14, "token LLM ↔ embedding"),
    (8, 12, "paella ↔ token LLM"),
    (0, 3, "cane abbaia ↔ canidi domestici"),
]
for i, j, descripcion in pares:
    sim = similitud_coseno(MATRIZ[i], MATRIZ[j])
    barra = "█" * int(sim * 20)
    print(f"  {descripcion:<40} {sim:.4f}  {barra}")

**Osservazioni** (modifica questa cella):

- La coppia con maggiore similarità è stata: ... con score ...
- La coppia con minore similarità è stata: ... con score ...
- Qualche risultato ti ha sorpreso? Quale e perché?
- A partire da quale soglia diresti che due frasi sono "correlate"?

---
## Cella 2 — Motore di ricerca semantica su un corpus di 80 frasi

Amplia il corpus a 80 frasi per rendere la ricerca più interessante.
Il motore di ricerca non usa alcun database: solo numpy.
È esattamente ciò che accade dentro un vector DB, ma reso visibile.

In [ ]:
# ── Corpus ampliato: 80 frasi in 8 categorie ───────────────────────────────

CORPUS = {
    "machine_learning": [
        "L'apprendimento supervisionato richiede dati etichettati",
        "L'overfitting si verifica quando il modello memorizza il set di addestramento",
        "La cross-validation valuta la capacità di generalizzazione del modello",
        "Gli alberi decisionali partizionano lo spazio delle feature",
        "Il gradient boosting combina più modelli deboli",
        "La regolarizzazione L2 penalizza i pesi grandi del modello",
        "La normalizzazione delle feature accelera la convergenza",
        "Le metriche di valutazione dipendono dal tipo di problema",
        "L'apprendimento per rinforzo ottimizza una politica tramite ricompense",
        "Le reti neurali apprendono rappresentazioni gerarchiche",
    ],
    "llm_rag": [
        "I modelli linguistici generano testo token per token",
        "Il RAG combina il recupero di documenti con la generazione di testo",
        "Gli embedding rappresentano il significato semantico in vettori",
        "La finestra di contesto limita quanto testo può elaborare il modello",
        "Il chunking divide i documenti in frammenti per indicizzarli",
        "La ricerca vettoriale trova documenti per similarità semantica",
        "Il prompt engineering migliora la qualità delle risposte dell'LLM",
        "Il fine-tuning adatta un modello preaddestrato a un compito specifico",
        "Le allucinazioni sono risposte plausibili ma scorrette",
        "La temperature controlla la casualità delle risposte generate",
    ],
    "python_programacion": [
        "Python è un linguaggio interpretato a tipizzazione dinamica",
        "I generatori in Python producono valori in modo differito",
        "I decoratori modificano il comportamento di funzioni e classi",
        "I context manager gestiscono le risorse con il protocollo with",
        "Le list comprehension sono più pythoniche dei cicli for",
        "I type hint migliorano la leggibilità e il tooling del codice",
        "asyncio permette la concorrenza tramite coroutine in Python",
        "Gli ambienti virtuali isolano le dipendenze di ogni progetto",
        "FastAPI usa Pydantic per validare automaticamente i dati in ingresso",
        "Le dataclass riducono il codice boilerplate per le classi di dati",
    ],
    "cloud_devops": [
        "Docker impacchetta le applicazioni con tutte le loro dipendenze",
        "Kubernetes orchestra i container in cluster di produzione",
        "CI/CD automatizza il processo di test e deploy del codice",
        "Le variabili d'ambiente memorizzano configurazioni sensibili in modo sicuro",
        "I microservizi scompongono le applicazioni in servizi indipendenti",
        "L'infrastructure as code permette di riprodurre gli ambienti esattamente",
        "Gli health check verificano che un servizio risponda correttamente",
        "Lo scaling orizzontale aggiunge più istanze invece di più potenza",
        "I log strutturati facilitano l'analisi e gli alert automatici",
        "Il rate limiting protegge i servizi da picchi di traffico inattesi",
    ],
    "bases_datos": [
        "Le transazioni ACID garantiscono la coerenza nei database relazionali",
        "Gli indici accelerano le query al costo di più spazio su disco",
        "Il modello relazionale organizza i dati in tabelle con chiavi esterne",
        "NoSQL sacrifica la coerenza per la scalabilità orizzontale",
        "PostgreSQL supporta tipi di dati avanzati come JSONB e array",
        "I database vettoriali indicizzano embedding per una ricerca rapida",
        "Lo sharding distribuisce i dati tra più nodi del database",
        "Le migrazioni dello schema devono essere reversibili in produzione",
        "Le query N+1 sono un antipattern comune negli ORM mal configurati",
        "Il connection pooling riutilizza le connessioni per migliorare le prestazioni",
    ],
    "seguridad": [
        "L'SQL injection è una delle vulnerabilità più pericolose nelle applicazioni web",
        "JWT permette di trasmettere informazioni verificate tra servizi in modo sicuro",
        "La crittografia end-to-end protegge i dati in transito e a riposo",
        "Il principio del minimo privilegio limita il danno in caso di violazione della sicurezza",
        "L'autenticazione a più fattori riduce significativamente il rischio di accesso non autorizzato",
        "Le chiavi API non devono mai essere hardcoded nel codice sorgente",
        "Il prompt injection è un attacco specifico delle applicazioni con LLM",
        "Il GDPR richiede una base giuridica per trattare i dati personali dei cittadini europei",
        "Le dipendenze obsolete sono un vettore d'attacco frequente",
        "L'analisi statica del codice rileva vulnerabilità prima del deploy",
    ],
    "negocio_freelance": [
        "La nicchia di mercato definisce chi aiuti e con quale problema specifico",
        "La proposta di valore spiega perché scegliere te rispetto alla concorrenza",
        "Il prezzo a valore si basa sul risultato per il cliente, non sulle ore",
        "Lo scoping del progetto evita lo scope creep e i malintesi successivi",
        "Un portfolio con demo pubblicate converte meglio di un repository vuoto",
        "Il retainer mensile garantisce entrate prevedibili al freelance",
        "LinkedIn è il canale principale di acquisizione B2B per servizi tecnici",
        "Il contratto deve specificare la proprietà intellettuale del codice consegnato",
        "La fatturazione da libero professionista richiede di includere correttamente IVA e imposte sul reddito",
        "L'NDA protegge le informazioni riservate del cliente durante il progetto",
    ],
    "medicina_salud": [
        "Gli studi clinici randomizzati sono il gold standard nella ricerca medica",
        "L'intelligenza artificiale sta trasformando la diagnostica per immagini",
        "La farmacocinetica studia come l'organismo elabora un farmaco",
        "I biomarcatori permettono di rilevare malattie in stadi precoci",
        "La telemedicina amplia l'accesso all'assistenza sanitaria nelle zone rurali",
        "Il microbioma intestinale influisce sul sistema immunitario e sull'umore",
        "La terapia genica permette di correggere mutazioni che causano malattie",
        "I dati sanitari sono soggetti a normative speciali sulla privacy",
        "La resistenza antibiotica è una minaccia globale emergente per la salute pubblica",
        "La medicina personalizzata adatta il trattamento al profilo genetico del paziente",
    ],
}

# Appiattisce il corpus in liste parallele
TEXTOS_CORPUS   = []
CATEGORIAS_CORPUS = []
for categoria, frases in CORPUS.items():
    TEXTOS_CORPUS.extend(frases)
    CATEGORIAS_CORPUS.extend([categoria] * len(frases))

print(f"Generando embedding per {len(TEXTOS_CORPUS)} documenti...")
EMBEDDINGS_CORPUS = generar_embeddings_batch(TEXTOS_CORPUS)   # shape (80, 1536)
print(f"Corpus indicizzato: {EMBEDDINGS_CORPUS.shape}")


# ── Motore di ricerca semantica ────────────────────────────────────────────────

def buscar_semantico(
    query: str,
    top_k: int = 5,
    umbral: float = 0.0
) -> list[dict]:
    """
    Cerca i documenti più simili semanticamente alla query.

    Flusso:
    1. Genera l'embedding della query
    2. Calcola la similarità coseno rispetto a tutti i documenti del corpus
    3. Restituisce i top_k più simili con score >= umbral
    """
    vector_query = generar_embedding(query)
    scores       = similitudes_batch(vector_query, EMBEDDINGS_CORPUS)
    indices_top  = np.argsort(scores)[::-1][:top_k]   # ordina in modo decrescente

    resultados = []
    for idx in indices_top:
        if scores[idx] >= umbral:
            resultados.append({
                "score":      float(scores[idx]),
                "texto":      TEXTOS_CORPUS[idx],
                "categoria":  CATEGORIAS_CORPUS[idx],
                "indice":     int(idx)
            })
    return resultados


def mostrar_resultados(query: str, resultados: list[dict]) -> None:
    print(f"\nQuery: '{query}'")
    print("-" * 70)
    for i, r in enumerate(resultados, 1):
        barra = "█" * int(r["score"] * 30)
        print(f"  {i}. [{r['score']:.4f}] {barra}")
        print(f"     {r['texto']}")
        print(f"     Categoria: {r['categoria']}")


# ── Test del motore di ricerca ──────────────────────────────────────────────────

QUERIES_PRUEBA = [
    "Come evito che il mio modello memorizzi i dati di addestramento?",
    "database per salvare vettori",
    "farmi pagare di più per i miei servizi di consulenza",
    "proteggere chiavi segrete nel codice",
    "cancro e trattamento genetico",              # termine medico
    "async await concorrenza",                     # mix inglese/italiano
]

for query in QUERIES_PRUEBA:
    resultados = buscar_semantico(query, top_k=3)
    mostrar_resultados(query, resultados)

**Osservazioni** (modifica questa cella):

- Query con i risultati più coerenti: ...
- Query con i risultati più sorprendenti: ...
- I risultati attraversano le categorie quando ha senso? Un esempio?
- Quale soglia di score useresti per filtrare i risultati irrilevanti?

---
## Cella 3 — Visualizzazione 2D degli embedding con PCA

Riduci 1536 dimensioni a 2 per visualizzare i cluster semantici.
Se gli embedding funzionano bene, gli 8 gruppi devono apparire raggruppati.

In [ ]:
# ── Riduzione PCA ─────────────────────────────────────────────────────────────

pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(EMBEDDINGS_CORPUS)   # shape (80, 2)

varianza = pca.explained_variance_ratio_
print(f"Varianza spiegata da PC1: {varianza[0]:.1%}")
print(f"Varianza spiegata da PC2: {varianza[1]:.1%}")
print(f"Totale conservata:        {varianza.sum():.1%}")
print(f"(Il {1 - varianza.sum():.1%} restante si perde nella riduzione)")


# ── Grafico a dispersione ─────────────────────────────────────────────────────

CATEGORIAS_UNICAS = list(CORPUS.keys())
COLORES           = cm.Set1(np.linspace(0, 1, len(CATEGORIAS_UNICAS)))
ICONOS = {
    "machine_learning":    "🤖",
    "llm_rag":             "💬",
    "python_programacion": "🐍",
    "cloud_devops":        "☁️",
    "bases_datos":         "🗄️",
    "seguridad":           "🔐",
    "negocio_freelance":   "💼",
    "medicina_salud":      "🏥",
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Pannello sinistro: punti con colore per categoria ──────────────────────────
ax = axes[0]
for i, categoria in enumerate(CATEGORIAS_UNICAS):
    mask = np.array(CATEGORIAS_CORPUS) == categoria
    puntos = embeddings_2d[mask]
    ax.scatter(puntos[:, 0], puntos[:, 1],
               c=[COLORES[i]], label=f"{ICONOS[categoria]} {categoria}",
               s=80, alpha=0.8, edgecolors="white", linewidth=0.5)

ax.set_title(f"Cluster semantici (PCA 2D)\nVarianza spiegata: {varianza.sum():.1%}",
             fontsize=12)
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.set_xlabel(f"PC1 ({varianza[0]:.1%} var.)")
ax.set_ylabel(f"PC2 ({varianza[1]:.1%} var.)")
ax.grid(True, alpha=0.3)

# ── Pannello destro: punti con etichetta di testo (solo indici visibili) ───────
ax2 = axes[1]
for i, categoria in enumerate(CATEGORIAS_UNICAS):
    mask_idx = [j for j, c in enumerate(CATEGORIAS_CORPUS) if c == categoria]
    for idx in mask_idx:
        ax2.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
                    c=[COLORES[i]], s=60, alpha=0.7)
        # Mostra solo le prime 3 parole del testo per non saturare
        etiqueta = " ".join(TEXTOS_CORPUS[idx].split()[:3]) + "..."
        ax2.annotate(etiqueta, (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
                     fontsize=5, alpha=0.7,
                     xytext=(3, 3), textcoords="offset points")

ax2.set_title("Con etichette di testo", fontsize=12)
ax2.set_xlabel(f"PC1 ({varianza[0]:.1%} var.)")
ax2.set_ylabel(f"PC2 ({varianza[1]:.1%} var.)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("embeddings_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafico salvato come embeddings_pca.png")

**Osservazioni** (modifica questa cella):

- Le categorie si separano bene in 2D? Quali si sovrappongono di più?
- La varianza totale spiegata è del ...%. È tanta o poca l'informazione che si perde?
- Quali categorie sono più vicine tra loro? Ha senso intuitivamente?
- Se usassi 3 dimensioni invece di 2, ti aspetteresti che le categorie si separino di più?

---
## Cella 4 — Comparativa: semantico vs parole chiave

Implementa una ricerca per parole chiave di base (TF semplice) e confrontala
con la ricerca semantica sulle stesse query.

Vedrai esattamente **quando fallisce ciascun approccio** e perché la settimana 8
(ricerca ibrida) combina entrambi.

In [ ]:
import re
from collections import Counter

# ── Motore di ricerca per parole chiave (TF semplice) ───────────────────────────────────

STOPWORDS_IT = {
    "il", "lo", "la", "i", "gli", "le", "un", "uno", "una", "di", "del",
    "in", "con", "per", "tra", "fra", "che", "è", "sono", "si", "suo", "al",
    "a", "e", "o", "non", "più", "come", "ma", "se", "questo", "questa",
    "questo", "quel", "quella", "quello", "ogni", "molto", "bene", "sta",
}

def tokenizar(texto: str) -> list[str]:
    """Tokenizzazione semplice: minuscolo, solo lettere e numeri."""
    tokens = re.findall(r"[a-zàèéìòù0-9]+", texto.lower())
    return [t for t in tokens if t not in STOPWORDS_IT and len(t) > 2]

def buscar_palabras_clave(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """Ricerca per conteggio di parole in comune (TF semplice, senza IDF)."""
    tokens_query = set(tokenizar(query))
    if not tokens_query:
        return []

    scores = []
    for i, texto in enumerate(TEXTOS_CORPUS):
        tokens_doc = set(tokenizar(texto))
        coincidencias = tokens_query & tokens_doc
        # Score: parole in comune / totale parole della query
        score = len(coincidencias) / len(tokens_query)
        if score > 0:
            scores.append({
                "score":         score,
                "texto":         texto,
                "categoria":     CATEGORIAS_CORPUS[i],
                "coincidencias": list(coincidencias)
            })

    scores.sort(key=lambda x: -x["score"])
    return scores[:top_k]


# ── Comparativa su 6 query pensate per mostrare le differenze ───────────

QUERIES_COMPARATIVA = [
    {
        "query":      "apprendimento automatico con supervisione",
        "descripcion": "Caso favorevole al semantico (sinonimi)"
    },
    {
        "query":      "overfitting regolarizzazione L2",
        "descripcion": "Caso favorevole alle parole chiave (termini tecnici esatti)"
    },
    {
        "query":      "fido abbaiare",
        "descripcion": "Il semantico trova 'cane', le parole chiave no"
    },
    {
        "query":      "GDPR dati personali",
        "descripcion": "Acronimo — come lo gestisce ciascun metodo?"
    },
    {
        "query":      "guadagnare lavorando da solo",
        "descripcion": "Metafora — il semantico dovrebbe trovare freelance"
    },
    {
        "query":      "Python async await coroutine",
        "descripcion": "Termini tecnici in inglese in un corpus in italiano"
    },
]

for caso in QUERIES_COMPARATIVA:
    query = caso["query"]
    print(f"\n{'='*70}")
    print(f"Query:        '{query}'")
    print(f"Descrizione:  {caso['descripcion']}")

    res_semantico = buscar_semantico(query, top_k=2)
    res_keywords  = buscar_palabras_clave(query, top_k=2)

    print(f"\n  [SEMANTICO]")
    if res_semantico:
        for r in res_semantico:
            print(f"    {r['score']:.4f} | {r['texto'][:60]}...")
    else:
        print("    (nessun risultato sopra soglia)")

    print(f"\n  [PAROLE CHIAVE]")
    if res_keywords:
        for r in res_keywords:
            coincide = ", ".join(r["coincidencias"])
            print(f"    {r['score']:.4f} | {r['texto'][:55]}... [{coincide}]")
    else:
        print("    (nessuna corrispondenza esatta)")

print(f"\n{'='*70}")
print("Conclusione: in quali casi ciascun metodo è superiore?")

**Tabella comparativa** (completa con le tue osservazioni):

| Caso | Vincitore | Perché |
|------|---------|--------|
| sinonimi | Semantico | ... |
| termini tecnici esatti | Parole chiave | ... |
| fido/cane | ... | ... |
| acronimi | ... | ... |
| metafore | ... | ... |
| inglese in corpus italiano | ... | ... |

**Conclusione:** perché la ricerca ibrida (settimana 8) è la soluzione corretta?

---
## Sfida finale — FAQ semantica

Costruisci un sistema di FAQ semantica per un'azienda software fittizia.

Dato un archivio di 20 domande frequenti con le loro risposte, quando l'utente
formula una domanda (con parole proprie, non necessariamente esatte),
il sistema trova la FAQ più simile e restituisce la sua risposta.

Questo è esattamente il caso d'uso più semplice di RAG: recuperare → restituire.

In [ ]:
# ── Archivio di FAQ ─────────────────────────────────────────────────────────────

FAQS = [
    {"pregunta": "Quanto costa il piano base?",
     "respuesta": "Il piano base costa 29€/mese e include fino a 3 utenti e 10 GB di spazio di archiviazione."},
    {"pregunta": "Posso disdire il mio abbonamento in qualsiasi momento?",
     "respuesta": "Sì, puoi disdire quando vuoi da Impostazioni > Abbonamento. Non ci sono penali."},
    {"pregunta": "Offrite sconti per le startup?",
     "respuesta": "Sì, le startup con meno di 2 anni e meno di 10 dipendenti ottengono uno sconto del 40% il primo anno."},
    {"pregunta": "Come reimposto la mia password?",
     "respuesta": "Nella schermata di login, clicca su 'Hai dimenticato la password?' e inserisci la tua email."},
    {"pregunta": "Dove sono ospitati i miei dati?",
     "respuesta": "I dati sono archiviati su server all'interno dell'UE (Francoforte, Irlanda), in conformità al GDPR."},
    {"pregunta": "L'applicazione ha un'integrazione con Slack?",
     "respuesta": "Sì, l'integrazione con Slack è disponibile dal piano Professionale in su."},
    {"pregunta": "Quanti utenti posso aggiungere?",
     "respuesta": "Base: 3 utenti. Professionale: 20 utenti. Enterprise: illimitati."},
    {"pregunta": "C'è una versione mobile?",
     "respuesta": "Sì, abbiamo app native per iOS e Android disponibili nei rispettivi store."},
    {"pregunta": "Come contatto il supporto tecnico?",
     "respuesta": "Apri un ticket su supporto.miapp.com oppure scrivici a supporto@miapp.com. Rispondiamo in meno di 4 ore."},
    {"pregunta": "Posso esportare i miei dati?",
     "respuesta": "Sì, puoi esportare tutti i tuoi dati in CSV o JSON da Impostazioni > Esporta dati."},
    {"pregunta": "Quali metodi di pagamento accettate?",
     "respuesta": "Accettiamo carta di credito/debito (Visa, Mastercard), PayPal e bonifico bancario per i piani Enterprise."},
    {"pregunta": "C'è un periodo di prova gratuito?",
     "respuesta": "Sì, 14 giorni di prova gratuita senza bisogno di carta di credito."},
    {"pregunta": "L'applicazione funziona senza connessione internet?",
     "respuesta": "C'è una modalità offline limitata nell'app mobile. Per la funzionalità completa serve la connessione."},
    {"pregunta": "Come aggiungo un nuovo membro al team?",
     "respuesta": "Vai su Impostazioni > Team > Invita utente. Inserisci l'email e assegna un ruolo."},
    {"pregunta": "Avete un'API pubblica?",
     "respuesta": "Sì, l'API REST è disponibile nel piano Professionale. Documentazione su api.miapp.com."},
    {"pregunta": "Come cambio il metodo di pagamento?",
     "respuesta": "In Impostazioni > Fatturazione > Metodo di pagamento puoi aggiornare o cambiare la tua carta."},
    {"pregunta": "La piattaforma è conforme al GDPR?",
     "respuesta": "Sì, siamo pienamente conformi al GDPR. Disponiamo di DPA, registro delle attività e strumenti per il consenso."},
    {"pregunta": "Posso usare il mio dominio personale?",
     "respuesta": "Sì, nel piano Enterprise puoi configurare un dominio personalizzato (es: app.tuaazienda.com)."},
    {"pregunta": "Cosa succede se supero il limite di spazio di archiviazione?",
     "respuesta": "Riceverai un avviso all'80% e al 95%. Al raggiungimento del 100% non potrai caricare altri file finché non amplii o liberi spazio."},
    {"pregunta": "Offrite formazione o onboarding?",
     "respuesta": "Sì, offriamo webinar settimanali gratuiti, tutorial video e onboarding personalizzato per i piani Enterprise."},
]


# ── Passo 1: indicizza le domande della FAQ ────────────────────────────────────
# TODO: genera gli embedding di tutte le domande della FAQ
# Suggerimento: usa generar_embeddings_batch() con la lista delle domande

preguntas_faq  = [faq["pregunta"]  for faq in FAQS]
respuestas_faq = [faq["respuesta"] for faq in FAQS]

print("Indicizzazione FAQ...")
# embeddings_faq = ...   # TODO
# print(f"FAQ indicizzate: {embeddings_faq.shape}")


# ── Passo 2: implementa il motore di ricerca della FAQ ─────────────────────────────────

def responder_faq(
    pregunta_usuario: str,
    umbral_confianza: float = 0.75
) -> dict:
    """
    Trova la FAQ più simile alla domanda dell'utente.

    Restituisce:
    {
        "respondida":  bool,
        "confianza":   float,
        "faq_encontrada": str,   # la domanda della FAQ che ha corrisposto
        "respuesta":   str,      # la risposta di quella FAQ
        "mensaje":     str        # messaggio per l'utente
    }

    Se la similarità < umbral_confianza, non risponde e suggerisce di contattare il supporto.
    """
    # TODO: implementa la funzione
    # 1. Genera l'embedding della domanda dell'utente
    # 2. Calcola le similarità rispetto a embeddings_faq
    # 3. Trova l'indice con maggiore similarità
    # 4. Se score >= umbral_confianza: restituisce la risposta
    # 5. Se score < umbral_confianza: restituisce messaggio di non trovato
    raise NotImplementedError("Implementa questa funzione")


# ── Passo 3: testa con domande formulate in modo diverso da quelle della FAQ ────
# Queste domande sono diverse da quelle della FAQ ma devono recuperare quella corretta

PREGUNTAS_TEST = [
    ("Quanto bisogna pagare ogni mese per il piano più economico?",         "prezzo piano base"),
    ("Voglio disdire, come faccio?",                                       "cancellare abbonamento"),
    ("Non ricordo la mia password di accesso",                             "reimpostare password"),
    ("I miei dati sono in Europa?",                                        "ubicazione dati / GDPR"),
    ("Funziona sul telefono?",                                             "versione mobile"),
    ("Quanto tempo ho per provarlo senza pagare?",                         "periodo di prova"),
    ("Avete sconti per le nuove aziende?",                                 "sconto startup"),
    ("Posso collegarlo ad altri strumenti tramite programmazione?",        "API pubblica"),
    ("Il sistema di archiviazione è quasi pieno, cosa succede?",           "limite di archiviazione"),
    ("Dove trovo la normativa sulla privacy?",                             "conformità GDPR"),
    ("Posso usare l'app quando non ho il wifi?",                           "modalità offline"),
    ("Avete supporto per le grandi aziende?",                              "non dovrebbe corrispondere direttamente"),
]

# Decommenta dopo aver implementato responder_faq()
# print("=" * 70)
# correctas = 0
# for pregunta, descripcion in PREGUNTAS_TEST:
#     resultado = responder_faq(pregunta)
#     icono = "✓" if resultado["respondida"] else "~"
#     print(f"\n{icono} [{descripcion}]")
#     print(f"  Query:      {pregunta}")
#     print(f"  FAQ match:  {resultado.get('faq_encontrada', 'N/D')[:60]}...")
#     print(f"  Confidenza: {resultado['confianza']:.4f}")
#     if resultado["respondida"]:
#         print(f"  Risposta:   {resultado['respuesta'][:80]}...")
#         correctas += 1
#
# print(f"\n{'='*70}")
# print(f"Recuperate: {correctas}/{len(PREGUNTAS_TEST)} ({correctas/len(PREGUNTAS_TEST):.0%})")

### Estensioni opzionali

1. **Cache degli embedding:** salva gli embedding del corpus in un file `.npy` per non rigenerarli a ogni esecuzione.

2. **Top-3 con confidenza:** invece di restituire solo il miglior risultato, restituisci i 3 più simili quando la confidenza del primo è tra 0.65 e 0.80, e lascia che sia l'utente a scegliere.

3. **UMAP al posto di PCA:** installa `umap-learn` e ripeti la visualizzazione della cella 3 con UMAP. I cluster si separano meglio?

In [ ]:
# Estensione 1: cache degli embedding su disco
import numpy as np
from pathlib import Path

def guardar_embeddings(embeddings: np.ndarray, ruta: str) -> None:
    """Salva gli embedding in formato .npy per riutilizzarli."""
    np.save(ruta, embeddings)
    print(f"Embedding salvati in {ruta} ({embeddings.shape})")

def cargar_o_generar_embeddings(textos: list[str], ruta_cache: str) -> np.ndarray:
    """Carica dalla cache se esiste; altrimenti genera e salva."""
    if Path(ruta_cache).exists():
        embeddings = np.load(ruta_cache)
        print(f"Embedding caricati dalla cache: {embeddings.shape}")
        return embeddings
    else:
        print("Cache non trovata. Generazione embedding...")
        embeddings = generar_embeddings_batch(textos)
        guardar_embeddings(embeddings, ruta_cache)
        return embeddings

# Usa la cache per il corpus della sfida
# embeddings_faq_cacheado = cargar_o_generar_embeddings(preguntas_faq, "faq_embeddings.npy")

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 05 — Settimana 5 del Piano di Formazione AI Engineering*